# MedFlow-XAI: GPU retraining runner (Colab / Kaggle)

Runs steps 2-3 of the retraining plan: full-dataset baseline + ResNet18 (+augmentation) for **the same number of epochs**, Grad-CAM, figures and results tables.

**Before you start:** Runtime -> Change runtime type -> GPU. Nothing here invents numbers: every metric is written by the repo's own scripts.

PostgreSQL is not needed on Colab; the scripts skip DB logging gracefully. Run the DB steps locally afterwards if you want `experiments` rows in PostgreSQL.

In [ ]:
import torch, subprocess
print("CUDA available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU found")
assert torch.cuda.is_available(), "Enable a GPU runtime first, otherwise this will take hours."


In [ ]:
# 1. Get the code and dependencies
!git clone https://github.com/Zoha257/MedFlow.git
%cd MedFlow
!pip install -q -r requirements.txt


In [ ]:
# 2. Download + validate PathMNIST (official MedMNIST source)
!python -m src.data.download
!python -m src.data.validate


In [ ]:
# 3. Baseline CNN, full dataset. EPOCHS must match the ResNet runs for a fair comparison.
EPOCHS = 10
BATCH = 1024   # if you get CUDA out-of-memory, try 256
!python -m src.models.train --epochs $EPOCHS --batch-size $BATCH


In [ ]:
# 4. ResNet18 and ResNet18 + augmentation, same budget
!python -m src.models.train_day3 --epochs $EPOCHS --batch-size $BATCH --experiment all


In [ ]:
# 5. Sanity-check before trusting anything. Collapse = model predicts only a few classes.
import json
cmp = json.load(open("results/model_comparison.json"))
for name, m in cmp.items():
    print(f"{name:14s} acc={m['accuracy']:.4f} f1={m['f1_macro']:.4f} epochs={m['epochs']} "
          f"max_samples={m.get('max_samples')} predicted_classes={m.get('num_predicted_classes')}")
bad = [n for n, m in cmp.items() if m.get('num_predicted_classes', 9) < 9 or m.get('max_samples') is not None]
assert not bad, f"Do not use these results (collapsed or smoke run): {bad}. Debug before continuing."


In [ ]:
# 6. Grad-CAM (served checkpoint = resnet18.pth), figures, and the generated results tables
!python -m src.models.generate_gradcam
!python -m src.research.generate_artifacts
!python -m src.research.render_results


In [ ]:
# 7. Package everything to copy back into the repository
!zip -qr medflow_retrain_outputs.zip models/resnet18.pth models/resnet18_aug.pth models/baseline_model.pth results paper/figures research data/processed/dataset_summary.json
from google.colab import files   # on Kaggle: download from the Output panel instead
files.download("medflow_retrain_outputs.zip")


## After downloading
1. Unzip over your local repository (overwrite `models/`, `results/`, `paper/figures/`, `research/`).
2. Bump `MEDFLOW_MODEL_VERSION` (e.g. `resnet18-v2`) in `.env` / `docker-compose.yml`.
3. Look at `paper/figures/correct_*.png` and `wrong_*.png`, then fill in the **Manual review** section of `research/gradcam_analysis.md`.
4. Rewrite the sections of `research/report.md` marked `REWRITE AFTER RETRAIN` using the numbers in the generated table.
5. `git rm --cached models/baseline_model.pth models/resnet18_aug.pth` (they are ignored from now on), then commit.